# Plymouth Geothermal Sensitivity Analysis


## 1. Imports


In [20]:
#import libraries
import gc, math
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box


## 2. Project and file paths


In [21]:
#configure paths
PROJECT_DIR = next(candidate for 
                   candidate in [Path.cwd().resolve(),*Path.cwd().resolve().parents] 
                   if (candidate / "01_Data").exists())

MODEL_OUTPUT_DIR = PROJECT_DIR / "01_Data/Processed/Geothermal/model_outputs"
ENERGY_PROCESSED_DIR = PROJECT_DIR / "01_Data/Processed/Energy"
OUT_DIR = PROJECT_DIR / "01_Data/Processed/Geothermal/sensitivity_analysis"
OUT_DIR.mkdir(parents=True,exist_ok=True)

CLOSED_LOOP_GPKG = MODEL_OUTPUT_DIR / "02_closed_loop_supply.gpkg"
CLOSED_LOOP_LAYER = "closed_loop_supply"
EXPORTED_75M_LAYER = "bhe_grid_75m"
ASSUMPTIONS_CSV = MODEL_OUTPUT_DIR / "02_closed_loop_model_assumptions.csv"
HEAT_DEMAND_GPKG = ENERGY_PROCESSED_DIR / "plymouth_heat_demand_2024.gpkg"
HEAT_DEMAND_LAYER = "lsoa_heat_demand_2024"

SPACING_SUMMARY_CSV = OUT_DIR / "spacing_edge_sensitivity_summary.csv"
FIXED_PARAM_CSV = OUT_DIR / "04_fixed_parameter_sensitivity.csv"

SPACINGS_M = (75.0,30.0,20.0,10.0)
USEFUL_HEAT_COL = "representative_useful_heat_mwh_year"
GROUND_PER_BHE_COL = "representative_ground_energy_mwh_per_borehole_year"
AREA_TOL_M2, NUMERIC_ATOL_MWH, NUMERIC_RTOL = 1e-7, 1e-6, 1e-10


## 3. Load inputs


In [22]:
#load model inputs
for path in [CLOSED_LOOP_GPKG,HEAT_DEMAND_GPKG,ASSUMPTIONS_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)

closed_loop = gpd.read_file(CLOSED_LOOP_GPKG,layer=CLOSED_LOOP_LAYER)
heat_demand = gpd.read_file(HEAT_DEMAND_GPKG,layer=HEAT_DEMAND_LAYER)
assumptions = pd.read_csv(ASSUMPTIONS_CSV)

required_inputs = [(closed_loop,["geology_polygon_id",GROUND_PER_BHE_COL,"geometry"],"closed-loop supply layer"),
                   (heat_demand,["LSOA_code","LSOA","total_useful_heat_mwh","geometry"],"heat-demand layer"),
                   (assumptions,["parameter","value"],"model assumptions CSV")]

for frame, columns, label in required_inputs:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{label} is missing expected columns: {missing}")

if closed_loop.crs != heat_demand.crs:
    heat_demand = heat_demand.to_crs(closed_loop.crs)
if closed_loop.crs is None or closed_loop.crs.to_epsg() != 27700:
    raise ValueError(f"Expected EPSG:27700 for metre-based calculations, found {closed_loop.crs}")

heat_demand = heat_demand[["LSOA_code","LSOA","total_useful_heat_mwh","geometry"]].copy()
heat_demand["geometry"] = heat_demand.geometry.make_valid()
if heat_demand["LSOA_code"].duplicated().any():
    raise ValueError("Duplicate LSOA codes detected in heat-demand layer.")

scop_row = assumptions.loc[assumptions["parameter"] == "heat_pump_scop","value"]
if len(scop_row) != 1:
    raise ValueError("Could not uniquely identify heat_pump_scop in assumptions CSV.")
SCOP = float(scop_row.iloc[0])
LSOA_BOUNDARY_UNION = heat_demand.geometry.boundary.union_all()

print("Closed-loop geology polygons:",len(closed_loop))
print("LSOAs:",len(heat_demand))
print("CRS:",closed_loop.crs)
print("SCOP loaded from production assumptions:",SCOP)


Closed-loop geology polygons: 328
LSOAs: 164
CRS: EPSG:27700
SCOP loaded from production assumptions: 3.2


## 4. Build BHE grids


In [23]:
#build the uniform BHE grid
def build_uniform_bhe_grid(spacing_m: float) -> gpd.GeoDataFrame:
    minx, miny, maxx, maxy = closed_loop.total_bounds
    x_start = np.floor(minx / spacing_m) * spacing_m + spacing_m / 2
    y_start = np.floor(miny / spacing_m) * spacing_m + spacing_m / 2
    x_coords = np.arange(x_start,maxx + spacing_m,spacing_m)
    y_coords = np.arange(y_start,maxy + spacing_m,spacing_m)
    xx, yy = np.meshgrid(x_coords,y_coords)

    candidate_grid = gpd.GeoDataFrame({"grid_id": np.arange(xx.size)},
                                       geometry=gpd.points_from_xy(xx.ravel(),yy.ravel()),
                                       crs=closed_loop.crs)
    grid = gpd.sjoin(candidate_grid,closed_loop[["geology_polygon_id",GROUND_PER_BHE_COL,"geometry"]],how="inner",predicate="within")
    grid = grid.drop(columns=["index_right"]).reset_index(drop=True)

    if grid["grid_id"].duplicated().any():
        raise ValueError(f"Duplicate BHE grid points detected at {spacing_m:g} m spacing.")

    grid["bhe_id"] = np.arange(1,len(grid) + 1)
    grid["grid_spacing_m"] = float(spacing_m)
    grid["easting_m"] = grid.geometry.x
    grid["northing_m"] = grid.geometry.y
    grid[USEFUL_HEAT_COL] = grid[GROUND_PER_BHE_COL] * SCOP / (SCOP - 1)

    return grid


In [24]:
#validate the generated 75 m grid
def validate_generated_75m_grid(generated_75m: gpd.GeoDataFrame) -> None:
    exported = gpd.read_file(CLOSED_LOOP_GPKG,layer=EXPORTED_75M_LAYER)
    required = ["bhe_id","easting_m","northing_m",USEFUL_HEAT_COL,"geometry"]
    missing = [column for column in required if column not in exported.columns]
    if missing:
        raise ValueError(f"exported 75 m BHE grid is missing expected columns: {missing}")

    if len(generated_75m) != len(exported):
        raise AssertionError(f"75 m count mismatch: regenerated={len(generated_75m):,}, exported={len(exported):,}")

    generated_xy = generated_75m[["easting_m","northing_m"]].sort_values(["easting_m","northing_m"]).to_numpy()
    exported_xy = exported[["easting_m","northing_m"]].sort_values(["easting_m","northing_m"]).to_numpy()
    if not np.array_equal(generated_xy,exported_xy):
        raise AssertionError("Regenerated 75 m BHE coordinates do not match production export.")

    generated_total = float(generated_75m[USEFUL_HEAT_COL].sum())
    exported_total = float(exported[USEFUL_HEAT_COL].sum())
    if not np.isclose(generated_total,exported_total,rtol=NUMERIC_RTOL,atol=NUMERIC_ATOL_MWH):
        raise AssertionError(f"75 m useful-heat mismatch: regenerated={generated_total}, exported={exported_total}")


## 5. Method A allocation


In [25]:
#resolve BHE points on LSOA boundaries
def resolve_unassigned_bhes(assigned: gpd.GeoDataFrame,spacing_m: float) -> gpd.GeoDataFrame:
    unassigned = assigned.loc[assigned["LSOA_code"].isna()].copy()
    half_cell = spacing_m / 2.0

    for _, row in unassigned.iterrows():
        bhe_id, point = row["bhe_id"], row.geometry
        touching = heat_demand.loc[heat_demand.geometry.intersects(point),["LSOA_code","LSOA","geometry"]].copy()
        if touching.empty:
            distances = heat_demand.geometry.distance(point)
            nearest_idx = distances.idxmin()
            raise ValueError(f"BHE {bhe_id} is unassigned and does not intersect any LSOA geometry. Nearest LSOA = {heat_demand.loc[nearest_idx,'LSOA']}; distance = {distances.loc[nearest_idx]:.12f} m.")

        if len(touching) == 1:
            chosen = touching.iloc[0]
        else:
            support_cell = box(point.x - half_cell,point.y - half_cell,point.x + half_cell,point.y + half_cell)
            touching["cell_overlap_m2"] = touching.geometry.intersection(support_cell).area
            max_area = touching["cell_overlap_m2"].max()
            winners = touching.loc[np.isclose(touching["cell_overlap_m2"],max_area,rtol=1e-12,atol=1e-9)].copy()
            chosen = winners.sort_values("LSOA_code").iloc[0]

        mask = assigned["bhe_id"] == bhe_id
        assigned.loc[mask,"LSOA_code"] = chosen["LSOA_code"]
        assigned.loc[mask,"LSOA"] = chosen["LSOA"]

    if assigned["LSOA_code"].isna().any():
        raise ValueError("One or more BHE points remain unassigned after boundary resolution.")
    return assigned


In [26]:
#assign BHE points using Method A
def method_a_point_assignment(bhe_grid: gpd.GeoDataFrame,spacing_m: float) -> tuple[pd.DataFrame,gpd.GeoDataFrame]:
    assigned = gpd.sjoin(bhe_grid[["bhe_id",USEFUL_HEAT_COL,"geometry"]],heat_demand[["LSOA_code","LSOA","geometry"]],
                         how="left",predicate="within")
    assigned = assigned.drop(columns=["index_right"]).reset_index(drop=True)

    if assigned["bhe_id"].duplicated().any():
        raise ValueError("Method A produced duplicate BHE assignments.")
    if assigned["LSOA_code"].isna().any():
        assigned = resolve_unassigned_bhes(assigned,spacing_m)
    if assigned["bhe_id"].nunique() != bhe_grid["bhe_id"].nunique():
        raise AssertionError("Method A does not preserve the original BHE count.")

    lsoa_supply = assigned.groupby(["LSOA_code","LSOA"],as_index=False).agg(
        borehole_count=("bhe_id","nunique"),representative_useful_heat_mwh_year=(USEFUL_HEAT_COL,"sum"))
    return lsoa_supply,assigned


## 6. Method B2 allocation


In [27]:
#allocate support cells using Method B2
def method_b2_domain_normalised(bhe_grid: gpd.GeoDataFrame,point_assignment: gpd.GeoDataFrame,spacing_m: float):
    half_cell = spacing_m / 2.0
    cell_area = spacing_m**2
    boundary_search = LSOA_BOUNDARY_UNION.buffer(spacing_m / math.sqrt(2.0) + 1e-9)
    boundary_search_gdf = gpd.GeoDataFrame({"_boundary_candidate": [1]},geometry=[boundary_search],crs=bhe_grid.crs)
    candidate_hits = gpd.sjoin(bhe_grid[["bhe_id","geometry"]],boundary_search_gdf,how="inner",predicate="within")
    candidate_ids = np.sort(candidate_hits["bhe_id"].unique())
    candidate_id_set = set(candidate_ids.tolist())

    candidate_cells = bhe_grid.loc[bhe_grid["bhe_id"].isin(candidate_ids),["bhe_id",USEFUL_HEAT_COL,"geometry"]].copy()
    candidate_cells["geometry"] = [box(point.x - half_cell,point.y - half_cell,point.x + half_cell,point.y + half_cell) for point in candidate_cells.geometry]
    candidate_cells = gpd.GeoDataFrame(candidate_cells,geometry="geometry",crs=bhe_grid.crs)
    fragments = gpd.overlay(candidate_cells,heat_demand[["LSOA_code","LSOA","geometry"]],how="intersection",keep_geom_type=False).reset_index(drop=True)
    fragments["fragment_area_m2"] = fragments.geometry.area
    fragments = fragments.loc[fragments["fragment_area_m2"] > AREA_TOL_M2].copy()

    represented_ids = set(fragments["bhe_id"].unique().tolist())
    missing_ids = candidate_id_set - represented_ids
    if missing_ids:
        raise ValueError(f"{len(missing_ids)} boundary-candidate cells produced no positive-area LSOA intersection.")

    support_area = fragments.groupby("bhe_id")["fragment_area_m2"].sum().rename("support_area_m2")
    fragments = fragments.merge(support_area,on="bhe_id",how="left")
    fragments["allocation_weight"] = fragments["fragment_area_m2"] / fragments["support_area_m2"]

    weight_sums = fragments.groupby("bhe_id")["allocation_weight"].sum()
    if not np.allclose(weight_sums.to_numpy(),1.0,rtol=1e-12,atol=1e-12):
        raise AssertionError("B2 allocation weights do not sum to 1 for every boundary cell.")

    fragments["allocated_useful_heat_mwh_year"] = fragments[USEFUL_HEAT_COL] * fragments["allocation_weight"]
    non_candidate = point_assignment.loc[~point_assignment["bhe_id"].isin(candidate_ids),["bhe_id","LSOA_code","LSOA",USEFUL_HEAT_COL]].copy()
    non_candidate["allocated_useful_heat_mwh_year"] = non_candidate[USEFUL_HEAT_COL]

    allocation = pd.concat([non_candidate[["bhe_id","LSOA_code","LSOA","allocated_useful_heat_mwh_year"]],
                            fragments[["bhe_id","LSOA_code","LSOA","allocated_useful_heat_mwh_year"]]],ignore_index=True)
    b2_supply = allocation.groupby(["LSOA_code","LSOA"],as_index=False).agg(
        representative_useful_heat_mwh_year=("allocated_useful_heat_mwh_year","sum"))
    actual_counts = point_assignment.groupby(["LSOA_code","LSOA"],as_index=False).agg(borehole_count=("bhe_id","nunique"))
    b2_supply = actual_counts.merge(b2_supply,on=["LSOA_code","LSOA"],how="outer")

    lsoa_counts = fragments.groupby("bhe_id")["LSOA_code"].nunique()
    all_lsoa_counts = pd.Series(1,index=bhe_grid["bhe_id"].to_numpy(),dtype="int64")
    all_lsoa_counts.loc[lsoa_counts.index] = lsoa_counts
    diagnostics = {"boundary_crossing_cells": int((all_lsoa_counts > 1).sum()),
                   "boundary_crossing_fraction_pct": float((all_lsoa_counts > 1).mean() * 100)}
    return b2_supply,diagnostics


## 7. Spacing sensitivity


In [28]:
#run Method A spacing and B2 allocation sensitivity
spacing_summary_rows = []

for spacing_m in SPACINGS_M:
    print(f"{spacing_m:g} m spacing")
    bhe_grid_spacing = build_uniform_bhe_grid(spacing_m)
    if np.isclose(spacing_m,75.0):
        validate_generated_75m_grid(bhe_grid_spacing)

    a_supply, point_assignment = method_a_point_assignment(bhe_grid_spacing,spacing_m)
    b2_supply, b2_diagnostics = method_b2_domain_normalised(bhe_grid_spacing,point_assignment,spacing_m)
    method_results = {}

    for method, supply in [("A",a_supply),("B2",b2_supply)]:
        lsoa = heat_demand[["LSOA_code","LSOA","total_useful_heat_mwh"]].merge(supply,on=["LSOA_code","LSOA"],how="left")
        lsoa[["borehole_count",USEFUL_HEAT_COL]] = lsoa[["borehole_count",USEFUL_HEAT_COL]].fillna(0)
        lsoa["borehole_count"] = lsoa["borehole_count"].astype(int)
        lsoa["supply_demand_ratio_pct"] = np.where(lsoa["total_useful_heat_mwh"] > 0,lsoa[USEFUL_HEAT_COL] / lsoa["total_useful_heat_mwh"] * 100,np.nan)
        lsoa["matched_heat_mwh_year"] = np.minimum(lsoa[USEFUL_HEAT_COL],lsoa["total_useful_heat_mwh"])
        lsoa["local_contribution_pct"] = np.where(lsoa["total_useful_heat_mwh"] > 0,lsoa["matched_heat_mwh_year"] / lsoa["total_useful_heat_mwh"] * 100,np.nan)
        lsoa["residual_demand_mwh_year"] = np.maximum(lsoa["total_useful_heat_mwh"] - lsoa[USEFUL_HEAT_COL],0)
        lsoa["surplus_mwh_year"] = np.maximum(lsoa[USEFUL_HEAT_COL] - lsoa["total_useful_heat_mwh"],0)

        source_total = float(bhe_grid_spacing[USEFUL_HEAT_COL].sum())
        allocated_total = float(lsoa[USEFUL_HEAT_COL].sum())
        if not np.isclose(source_total,allocated_total,rtol=NUMERIC_RTOL,atol=NUMERIC_ATOL_MWH):
            raise AssertionError(f"Method {method} failed conservation at {spacing_m:g} m spacing.")
        method_results[method] = lsoa

    a_lsoa = method_results["A"]
    b2_lsoa = method_results["B2"]
    comparison = a_lsoa.rename(columns={USEFUL_HEAT_COL: f"A_{USEFUL_HEAT_COL}",
                                        "local_contribution_pct": "A_local_contribution_pct"})
    comparison = comparison.merge(
        b2_lsoa[["LSOA_code",USEFUL_HEAT_COL,"local_contribution_pct"]].rename(
            columns={USEFUL_HEAT_COL: f"B2_{USEFUL_HEAT_COL}","local_contribution_pct": "B2_local_contribution_pct"}),
        on="LSOA_code",how="inner")
    comparison["supply_change_mwh"] = comparison[f"B2_{USEFUL_HEAT_COL}"] - comparison[f"A_{USEFUL_HEAT_COL}"]
    comparison["absolute_supply_change_mwh"] = comparison["supply_change_mwh"].abs()
    comparison["supply_change_pct"] = np.where(comparison[f"A_{USEFUL_HEAT_COL}"] != 0,comparison["supply_change_mwh"] / comparison[f"A_{USEFUL_HEAT_COL}"] * 100,np.nan)
    comparison["coverage_change_pp"] = comparison["B2_local_contribution_pct"] - comparison["A_local_contribution_pct"]
    comparison["absolute_coverage_change_pp"] = comparison["coverage_change_pp"].abs()

    a_supply_order = comparison.sort_values(f"A_{USEFUL_HEAT_COL}",ascending=False)["LSOA_code"].head(10).tolist()
    b2_supply_order = comparison.sort_values(f"B2_{USEFUL_HEAT_COL}",ascending=False)["LSOA_code"].head(10).tolist()

    spacing_summary_rows.append({
        "spacing_m": spacing_m,
        "nominal_grid_density_bhe_km2": 1_000_000 / spacing_m**2,
        "actual_bhe_points": len(bhe_grid_spacing),
        "method_a_supply_gwh_year": a_lsoa[USEFUL_HEAT_COL].sum() / 1000,
        "method_a_matched_heat_gwh_year": a_lsoa["matched_heat_mwh_year"].sum() / 1000,
        "method_a_coverage_pct": a_lsoa["matched_heat_mwh_year"].sum() / a_lsoa["total_useful_heat_mwh"].sum() * 100,
        "method_b2_supply_gwh_year": b2_lsoa[USEFUL_HEAT_COL].sum() / 1000,
        "method_b2_matched_heat_gwh_year": b2_lsoa["matched_heat_mwh_year"].sum() / 1000,
        "method_b2_coverage_pct": b2_lsoa["matched_heat_mwh_year"].sum() / b2_lsoa["total_useful_heat_mwh"].sum() * 100,
        "matched_heat_change_gwh_B2_minus_A": (b2_lsoa["matched_heat_mwh_year"].sum() - a_lsoa["matched_heat_mwh_year"].sum()) / 1000,
        "coverage_change_pp_B2_minus_A": b2_lsoa["matched_heat_mwh_year"].sum() / b2_lsoa["total_useful_heat_mwh"].sum() * 100 - a_lsoa["matched_heat_mwh_year"].sum() / a_lsoa["total_useful_heat_mwh"].sum() * 100,
        "max_abs_lsoa_supply_change_gwh": comparison["absolute_supply_change_mwh"].max() / 1000,
        "median_abs_lsoa_supply_change_gwh": comparison["absolute_supply_change_mwh"].median() / 1000,
        "max_abs_lsoa_coverage_change_pp": comparison["absolute_coverage_change_pp"].max(),
        "median_abs_lsoa_coverage_change_pp": comparison["absolute_coverage_change_pp"].median(),
        "boundary_crossing_cells": b2_diagnostics["boundary_crossing_cells"],
        "boundary_crossing_fraction_pct": b2_diagnostics["boundary_crossing_fraction_pct"],
        "top10_supply_same_set": set(a_supply_order) == set(b2_supply_order),
        "top10_supply_same_order": a_supply_order == b2_supply_order})

    del bhe_grid_spacing, point_assignment, a_supply, b2_supply, method_results, comparison
    gc.collect()

spacing_summary = pd.DataFrame(spacing_summary_rows).round(4)
spacing_summary


75 m spacing
30 m spacing
20 m spacing
10 m spacing


,spacing_m,nominal_grid_density_bhe_km2,actual_bhe_points,method_a_supply_gwh_year,method_a_matched_heat_gwh_year,method_a_coverage_pct,method_b2_supply_gwh_year,method_b2_matched_heat_gwh_year,method_b2_coverage_pct,matched_heat_change_gwh_B2_minus_A,coverage_change_pp_B2_minus_A,max_abs_lsoa_supply_change_gwh,median_abs_lsoa_supply_change_gwh,max_abs_lsoa_coverage_change_pp,median_abs_lsoa_coverage_change_pp,boundary_crossing_cells,boundary_crossing_fraction_pct,top10_supply_same_set,top10_supply_same_order
0,75.0,177.7778,14056,398.4074,355.8969,44.7769,398.4074,356.0872,44.8009,0.1903,0.0239,0.1559,0.0334,4.3042,0.629,4418,31.4314,True,False
1,30.0,1111.1111,87808,2488.3215,776.0324,97.6360,2488.3215,776.0899,97.6432,0.0575,0.0072,0.2717,0.0554,2.6545,0.000,12060,13.7345,True,True
2,20.0,2500.0000,197501,5596.8091,794.8222,100.0000,5596.8091,794.8222,100.0000,0.0000,0.0000,0.3250,0.0614,0.0000,0.000,18410,9.3215,True,True
3,10.0,10000.0000,790185,22391.9704,794.8222,100.0000,22391.9704,794.8222,100.0000,0.0000,0.0000,0.4915,0.0896,0.0000,0.000,37330,4.7242,True,True


## 8. Save spacing sensitivity output


In [29]:
#save spacing sensitivity table
spacing_summary.to_csv(SPACING_SUMMARY_CSV,index=False)


## 9. Depth and Rb sensitivity


In [30]:
#load fixed-parameter sensitivity inputs
bhe_grid_75m = gpd.read_file(CLOSED_LOOP_GPKG,layer=EXPORTED_75M_LAYER)
baseline_demand_mwh = heat_demand["total_useful_heat_mwh"].to_numpy()

GROUND_TEMPERATURE_C = 14.05
BOREHOLE_DEPTH_M = 150.0
BOREHOLE_RADIUS_M = 0.075
BOREHOLE_THERMAL_RESISTANCE_MK_W = 0.068
MIN_FLUID_TEMPERATURE_C = -2
HEATING_SEASON_DAYS = 182.0
SIMULATION_LIFETIME_YEARS = 50.0
HEAT_PUMP_SCOP = SCOP
ONE_YEAR_SECONDS = 365.25 * 24 * 3600
HEATING_SEASON_SECONDS = HEATING_SEASON_DAYS * 24 * 3600
SIMULATION_SECONDS = SIMULATION_LIFETIME_YEARS * ONE_YEAR_SECONDS
TC_RATIO = HEATING_SEASON_SECONDS / ONE_YEAR_SECONDS


In [31]:
#calculate geothermal supply with parameter overrides
def calculate_geothermal_supply_mwh_override(lambda_w_mk,volumetric_heat_capacity_mj_m3k,
                                             ground_temperature_c=GROUND_TEMPERATURE_C,
                                             borehole_thermal_resistance_mk_w=BOREHOLE_THERMAL_RESISTANCE_MK_W,
                                             borehole_depth_m=BOREHOLE_DEPTH_M,heat_pump_scop=HEAT_PUMP_SCOP):
    lam = np.asarray(lambda_w_mk,dtype=float)
    cv = np.asarray(volumetric_heat_capacity_mj_m3k,dtype=float)
    if lam.shape != cv.shape:
        raise ValueError("Conductivity and volumetric heat-capacity arrays must match.")
    if np.any(lam <= 0):
        raise ValueError("Thermal conductivity must be positive.")
    if np.any(cv <= 0):
        raise ValueError("Volumetric heat capacity must be positive.")

    Cv = cv * 1_000_000
    alpha = lam / Cv
    u_s = BOREHOLE_RADIUS_M**2 / (4 * alpha * SIMULATION_SECONDS)
    u_c = BOREHOLE_RADIUS_M**2 / (4 * alpha * HEATING_SEASON_SECONDS)
    denominator = (-0.619 * TC_RATIO * np.log(u_s) + (0.532 * TC_RATIO - 0.962) * np.log(u_c)
                   - 0.455 * TC_RATIO - 1.619 + 4 * np.pi * lam * borehole_thermal_resistance_mk_w)
    q_bhe_w = (8 * (ground_temperature_c - MIN_FLUID_TEMPERATURE_C) * lam * borehole_depth_m * TC_RATIO / denominator)
    ground_energy_mwh_per_bhe = q_bhe_w * 8760 / 1_000_000
    return ground_energy_mwh_per_bhe * heat_pump_scop / (heat_pump_scop - 1)


In [32]:
#cache the 75 m BHE-to-LSOA assignment
_, BHE_LSOA_75M = method_a_point_assignment(bhe_grid_75m,75.0)

def matched_contribution_pct(bhe_supply_mwh):
    if len(bhe_supply_mwh) != len(BHE_LSOA_75M):
        raise ValueError("BHE supply values must match the cached 75 m BHE assignment.")
    bhe_lsoa = BHE_LSOA_75M[["bhe_id","LSOA_code"]].copy()
    bhe_lsoa["supply_mwh"] = np.asarray(bhe_supply_mwh)
    lsoa_supply = (bhe_lsoa.groupby("LSOA_code")["supply_mwh"].sum()
                   .reindex(heat_demand["LSOA_code"],fill_value=0).to_numpy())
    matched = np.minimum(lsoa_supply,baseline_demand_mwh)
    return matched.sum() / baseline_demand_mwh.sum() * 100,lsoa_supply.sum()


In [33]:
#define local appraisal scenarios
LOCAL_APPRAISAL_SCENARIOS = {
    "Borehole depth L (m)": {"kwarg_name": "borehole_depth_m","baseline_value": BOREHOLE_DEPTH_M,
                              "alternative_value": 200.0,"alternative_label": "Plymouth appraisal (200 m)"},
    "Borehole thermal resistance Rb (m K/W)": {"kwarg_name": "borehole_thermal_resistance_mk_w","baseline_value": BOREHOLE_THERMAL_RESISTANCE_MK_W,
                                                 "alternative_value": 0.075,"alternative_label": "Plymouth appraisal (0.075 m K/W)"}}


## 10. Run fixed-parameter sensitivity


In [34]:
#calculate baseline fixed-parameter result
representative_lambda = bhe_grid_75m["thermal_conductivity_w_mk"].to_numpy()
volumetric_cv = bhe_grid_75m["volumetric_heat_capacity_mj_m3k"].to_numpy()
baseline_supply = calculate_geothermal_supply_mwh_override(representative_lambda,volumetric_cv)
baseline_matched_pct, baseline_total_supply_mwh = matched_contribution_pct(baseline_supply)
production_supply = bhe_grid_75m[USEFUL_HEAT_COL].sum()

if not np.isclose(baseline_total_supply_mwh,production_supply):
    raise AssertionError("Baseline fixed-parameter result does not match production supply.")


In [35]:
#run local appraisal scenarios
local_scenario_rows = []

for label, scenario in LOCAL_APPRAISAL_SCENARIOS.items():
    alternative_supply = calculate_geothermal_supply_mwh_override(
        representative_lambda,volumetric_cv,
        **{scenario["kwarg_name"]: scenario["alternative_value"]})
    alternative_matched_pct, alternative_total_supply_mwh = matched_contribution_pct(alternative_supply)
    local_scenario_rows.append({
        "parameter": label,
        "baseline_value": scenario["baseline_value"],
        "alternative_value": scenario["alternative_value"],
        "alternative_source": scenario["alternative_label"],
        "supply_baseline_gwh": baseline_total_supply_mwh / 1_000,
        "supply_alternative_gwh": alternative_total_supply_mwh / 1_000,
        "matched_pct_baseline": baseline_matched_pct,
        "matched_pct_alternative": alternative_matched_pct,
        "matched_pct_difference": alternative_matched_pct - baseline_matched_pct})

fixed_parameter_sensitivity = pd.DataFrame(local_scenario_rows)
fixed_parameter_sensitivity["abs_matched_pct_difference"] = fixed_parameter_sensitivity["matched_pct_difference"].abs()
fixed_parameter_sensitivity = (fixed_parameter_sensitivity.sort_values("abs_matched_pct_difference",ascending=False)
                                .drop(columns="abs_matched_pct_difference").reset_index(drop=True))
numeric_cols = [column for column in fixed_parameter_sensitivity.columns if column not in ("parameter","alternative_source")]
fixed_parameter_sensitivity[numeric_cols] = fixed_parameter_sensitivity[numeric_cols].round(4)
fixed_parameter_sensitivity


,parameter,baseline_value,alternative_value,alternative_source,supply_baseline_gwh,supply_alternative_gwh,matched_pct_baseline,matched_pct_alternative,matched_pct_difference
0,Borehole depth L (m),150.000,200.000,Plymouth appraisal (200 m),398.4074,531.2099,44.7769,55.6243,10.8474
1,Borehole thermal resistance Rb (m K/W),0.068,0.075,Plymouth appraisal (0.075 m K/W),398.4074,390.4241,44.7769,44.0372,-0.7397


## 11. Save fixed-parameter output


In [36]:
#save fixed-parameter sensitivity table
fixed_parameter_sensitivity.to_csv(FIXED_PARAM_CSV,index=False)
